In [3]:
import numpy as np
import pandas as pd

import os
import sys
from pathlib import Path

rootdir = ("/home/jschleic/EquiReact/src/")#os.path.dirname(os.path.abspath(__file__))

os.chdir(f"{rootdir}/..")

sys.path.insert(0, rootdir)


In [4]:
from process.dataloader_juliette import Juliette

from process.splitter import split_dataset

/home/jschleic/miniconda3/envs/equireact-kuma/lib/python3.10/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
#need to define the missing variables (copy paste from equireact)
#--train_frac 0.8 \
#--combine_mode diff \
#--distance_emb_dim 64 \
#--dropout_p 0.0 \
#--graph_mode energy \
#--lr 0.001 \
#--max_neighbors 50 \
#--n_conv_layers 2 \
#--n_s 64 \
#--n_v 16 \
#--radius 2.5 \
#--sum_mode both \
#--weight_decay 0.00001 \
#--two_layers_atom_diff \
#--splitter random \
#--noH \
#--invariant \
#--atom_mapping \

subset=None
training_fractions = [0.8]
process=False
CV=0
splitter='random'
atom_mapping=False
noH=False

DATASET='juliette:cmd:int:int'
dataset = DATASET.split(':')[0]
react = DATASET.split(':')[1]
geom_r = DATASET.split(':')[2]
geom_p = DATASET.split(':')[3]

#dataset would need to be included in splitter.py if we choose something else than a random split!
data = Juliette(process=process, atom_mapping=atom_mapping, noH=noH, geometry=geom_r, reaction=react, geometry_p=geom_p)

Loading data into memory...
dataset_prefix='CMD_TS_smiles_ok.int'
Processed data not found, processing data...
Processing xyz files and saving coords to data/juliette/processed//


making graphs:   0%|          | 0/789 [00:00<?, ?it/s]/home/jschleic/EquiReact/process/feature.py:570: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  geom_node_feat = torch.tensor(
making graphs: 100%|██████████| 789/789 [00:47<00:00, 16.63it/s]


Saved graphs to data/juliette/processed//CMD_TS_smiles_ok.int.v3.reactants_graphs.pt and data/juliette/processed//CMD_TS_smiles_ok.int.v3.products_graphs.pt


In [6]:
tr_indices, te_indices, val_indices, indices = split_dataset(nreactions=data.nreactions, splitter=splitter,tr_frac=max(training_fractions),dataset=dataset, subset=subset)

Using random splits


In [7]:
# split the csv file into train, val, test
if "cmd" in DATASET:
    csv_name = f"data/juliette/CMD_TS_smiles_ok.csv"
elif "irb" in DATASET:
    csv_name = f"data/juliette/IRB_TS_smiles_ok.csv"
else:
    raise ValueError("Dataset not recognized")

df = pd.read_csv(csv_name)

In [8]:
df_tr = df.iloc[tr_indices]
df_val = df.iloc[val_indices]
df_te = df.iloc[te_indices]

In [9]:
df_tr_te = pd.concat([df_tr, df_te])
df_tr_te.to_csv(f"{csv_name[:-4]}_train_test.csv", index=False)
df_val.to_csv(f"{csv_name[:-4]}_val.csv", index=False)